# Notebook 45: Longitudinal Evaluation

This notebook demonstrates long-running evaluation tracking using `multigen.eval_longitudinal`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `GoldenDataset` — add examples, filter by tags, sample
- `RegressionSuite` — run mock agent, inspect `RegressionResult`
- `VersionScoreHistory` — track scores across versions with trend/delta
- `DriftAlerter` — fire `DriftAlert` when scores drop beyond a threshold
- `LongitudinalEvalManager` facade — end-to-end workflow

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.eval_longitudinal`

In [ ]:
import random
import statistics
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

random.seed(42)

# ---------------------------------------------------------------------------
# 1. GoldenDataset
# ---------------------------------------------------------------------------

@dataclass
class GoldenExample:
    id: str
    input: str
    expected: str
    tags: List[str] = field(default_factory=list)


class GoldenDataset:
    def __init__(self, name: str):
        self.name = name
        self._examples: List[GoldenExample] = []

    def add(self, id: str, input: str, expected: str, tags: List[str] = None):
        self._examples.append(GoldenExample(id=id, input=input, expected=expected, tags=tags or []))

    def filter(self, tag: str) -> List[GoldenExample]:
        return [e for e in self._examples if tag in e.tags]

    def sample(self, n: int, seed: int = 0) -> List[GoldenExample]:
        rng = random.Random(seed)
        pool = list(self._examples)
        rng.shuffle(pool)
        return pool[:n]

    def __len__(self):
        return len(self._examples)

print('GoldenDataset defined.')


## Section 1 — GoldenDataset

In [ ]:
dataset = GoldenDataset('qa_v1')
dataset.add('q1', 'What is 2+2?',       '4',     tags=['math', 'easy'])
dataset.add('q2', 'Summarise paragraph X', 'Summary X', tags=['summarisation'])
dataset.add('q3', 'Translate "hello"',   'hola',  tags=['translation', 'easy'])
dataset.add('q4', 'What is the capital of France?', 'Paris', tags=['geography', 'easy'])
dataset.add('q5', 'Solve integral of x^2', 'x^3/3 + C', tags=['math', 'hard'])
dataset.add('q6', 'Classify email tone',  'positive', tags=['classification'])

print(f'Total examples: {len(dataset)}')

easy = dataset.filter('easy')
print(f'Easy examples ({len(easy)}): {[e.id for e in easy]}')

math = dataset.filter('math')
print(f'Math examples ({len(math)}): {[e.id for e in math]}')

sampled = dataset.sample(3, seed=7)
print(f'\nSampled 3 examples: {[e.id for e in sampled]}')


## Section 2 — RegressionSuite

In [ ]:
@dataclass
class RegressionResult:
    version: str
    scores: List[float]

    @property
    def mean_score(self) -> float:
        return round(statistics.mean(self.scores), 4) if self.scores else 0.0

    @property
    def pass_rate(self) -> float:
        """Fraction of examples with score >= 0.7"""
        return round(sum(s >= 0.7 for s in self.scores) / len(self.scores), 4) if self.scores else 0.0

    @property
    def trend(self) -> str:
        if len(self.scores) < 2:
            return 'stable'
        return 'improving' if self.scores[-1] > self.scores[0] else 'declining'


class RegressionSuite:
    def __init__(self, dataset: GoldenDataset, scorer: Callable = None):
        self.dataset = dataset
        self.scorer = scorer or self._default_scorer

    @staticmethod
    def _default_scorer(example: GoldenExample, prediction: str) -> float:
        """Simple exact-match scorer returning 1.0 or partial overlap."""
        if prediction.strip().lower() == example.expected.strip().lower():
            return 1.0
        overlap = len(set(prediction.lower().split()) & set(example.expected.lower().split()))
        return round(min(overlap / max(len(example.expected.split()), 1), 0.9), 3)

    def run(self, agent_fn: Callable[[str], str], version: str) -> RegressionResult:
        scores = []
        for ex in self.dataset._examples:
            prediction = agent_fn(ex.input)
            score = self.scorer(ex, prediction)
            scores.append(score)
        return RegressionResult(version=version, scores=scores)


# Mock agents with different quality levels
def mock_agent_v1(text: str) -> str:
    """Poor agent — barely correct."""
    answers = {'What is 2+2?': '4', 'What is the capital of France?': 'Paris'}
    return answers.get(text, 'I do not know')

def mock_agent_v2(text: str) -> str:
    """Better agent."""
    answers = {
        'What is 2+2?': '4',
        'What is the capital of France?': 'Paris',
        'Translate "hello"': 'hola',
        'Classify email tone': 'positive',
    }
    return answers.get(text, 'unknown answer')


suite = RegressionSuite(dataset)
result_v1 = suite.run(mock_agent_v1, version='v1')
result_v2 = suite.run(mock_agent_v2, version='v2')

for r in [result_v1, result_v2]:
    print(f'Version {r.version}:')
    print(f'  scores     = {[round(s, 2) for s in r.scores]}')
    print(f'  mean_score = {r.mean_score}')
    print(f'  pass_rate  = {r.pass_rate}')
    print(f'  trend      = {r.trend}')
    print()


## Section 3 — VersionScoreHistory

In [ ]:
class VersionScoreHistory:
    def __init__(self):
        self._records: List[Dict] = []  # [{version, score}, ...]

    def record(self, version: str, score: float):
        self._records.append({'version': version, 'score': score})

    def trend(self) -> str:
        if len(self._records) < 2:
            return 'stable'
        first = self._records[0]['score']
        last = self._records[-1]['score']
        if last > first + 0.02:
            return 'improving'
        if last < first - 0.02:
            return 'declining'
        return 'stable'

    def delta(self, v_from: str, v_to: str) -> float:
        scores = {r['version']: r['score'] for r in self._records}
        return round(scores[v_to] - scores[v_from], 4)

    def moving_average(self, window: int = 3) -> List[float]:
        scores = [r['score'] for r in self._records]
        result = []
        for i in range(len(scores)):
            chunk = scores[max(0, i - window + 1): i + 1]
            result.append(round(statistics.mean(chunk), 4))
        return result


history = VersionScoreHistory()
history.record('v1', 0.55)
history.record('v2', 0.63)
history.record('v3', 0.71)

print('Version score history:')
for r in history._records:
    print(f'  {r["version"]}: {r["score"]}')

print(f'\ntrend()           : {history.trend()}')
print(f'delta(v1 -> v3)   : {history.delta("v1", "v3")}')
print(f'delta(v2 -> v3)   : {history.delta("v2", "v3")}')
print(f'moving_average(2) : {history.moving_average(window=2)}')
print(f'moving_average(3) : {history.moving_average(window=3)}')


## Section 4 — DriftAlerter

In [ ]:
@dataclass
class DriftAlert:
    version: str
    score: float
    baseline: float
    drop: float
    threshold: float

    def __str__(self):
        return (
            f'DriftAlert: version={self.version} score={self.score:.3f} '
            f'(baseline={self.baseline:.3f}, drop={self.drop:.3f}, threshold={self.threshold:.3f})'
        )


class DriftAlerter:
    def __init__(self, threshold: float = 0.05, baseline_version: str = None):
        self.threshold = threshold
        self.baseline_version = baseline_version
        self._history = VersionScoreHistory()
        self._alerts: List[DriftAlert] = []

    def observe(self, version: str, score: float) -> Optional[DriftAlert]:
        self._history.record(version, score)
        records = self._history._records
        if len(records) < 2:
            return None
        # Compare against the designated baseline or the first recorded version
        if self.baseline_version:
            scores_map = {r['version']: r['score'] for r in records}
            if self.baseline_version not in scores_map:
                return None
            baseline_score = scores_map[self.baseline_version]
        else:
            baseline_score = records[0]['score']

        drop = baseline_score - score
        if drop >= self.threshold:
            alert = DriftAlert(
                version=version, score=score,
                baseline=baseline_score, drop=round(drop, 4),
                threshold=self.threshold,
            )
            self._alerts.append(alert)
            return alert
        return None

    @property
    def alerts(self) -> List[DriftAlert]:
        return list(self._alerts)


alerter = DriftAlerter(threshold=0.05, baseline_version='v1')

score_stream = [('v1', 0.80), ('v2', 0.79), ('v3', 0.75), ('v4', 0.72), ('v5', 0.65)]

print('Observing score stream (baseline=v1, threshold=0.05):\n')
for version, score in score_stream:
    alert = alerter.observe(version, score)
    flag = f'  ** {alert}' if alert else ''
    print(f'  {version}: {score:.2f}{flag}')

print(f'\nTotal alerts fired: {len(alerter.alerts)}')
for a in alerter.alerts:
    print(f'  {a}')


## Section 5 — LongitudinalEvalManager Facade

In [ ]:
class LongitudinalEvalManager:
    """Facade combining GoldenDataset, RegressionSuite, VersionScoreHistory, DriftAlerter."""

    def __init__(self, dataset_name: str, drift_threshold: float = 0.05):
        self.dataset = GoldenDataset(dataset_name)
        self._suite: Optional[RegressionSuite] = None
        self._score_history = VersionScoreHistory()
        self._alerter: Optional[DriftAlerter] = None
        self._drift_threshold = drift_threshold

    def create_suite(self, scorer: Callable = None):
        self._suite = RegressionSuite(self.dataset, scorer)
        print(f'[EvalManager] Suite created over {len(self.dataset)} examples.')

    def run(self, agent_fn: Callable, version: str) -> RegressionResult:
        result = self._suite.run(agent_fn, version)
        return result

    def record(self, version: str, score: float):
        self._score_history.record(version, score)
        if self._alerter is None:
            # First record becomes baseline
            self._alerter = DriftAlerter(threshold=self._drift_threshold,
                                          baseline_version=version)

    def check_drift(self, version: str, score: float) -> Optional[DriftAlert]:
        return self._alerter.observe(version, score)


# Build dataset
mgr = LongitudinalEvalManager('prod_qa', drift_threshold=0.05)
for item in [
    ('e1', 'Capital of Japan?',      'Tokyo',  ['geo']),
    ('e2', 'What is 5*5?',           '25',     ['math']),
    ('e3', 'Translate "goodbye"',    'adios',  ['translation']),
    ('e4', 'Best sorting algorithm?', 'merge sort', ['cs']),
]:
    mgr.dataset.add(*item)

mgr.create_suite()

# Simulate 3 agent releases
agents = {
    'v1': lambda t: {'Capital of Japan?': 'Tokyo', 'What is 5*5?': '25'}.get(t, 'unknown'),
    'v2': lambda t: {'Capital of Japan?': 'Tokyo', 'What is 5*5?': '25',
                     'Translate "goodbye"': 'adios'}.get(t, 'unknown'),
    'v3': lambda t: 'unknown',   # regression — agent broke
}

print()
baseline_set = False
for ver, agent in agents.items():
    result = mgr.run(agent, ver)
    mgr.record(ver, result.mean_score)
    if not baseline_set:
        baseline_set = True
    else:
        alert = mgr.check_drift(ver, result.mean_score)
        if alert:
            print(f'  !! {alert}')
    print(f'  {ver}: mean={result.mean_score:.3f}  pass_rate={result.pass_rate:.3f}')

print(f'\nOverall trend: {mgr._score_history.trend()}')
